# LSM companion runtime (Google Colab)

Ноутбук-компаньон интерактивных лаб **LLM Serving Mastery**.

**Как пользоваться:** запусти ячейку ниже → скопируй напечатанные **URL** и **токен** → вставь их в панель «Удалённый рантайм» на странице лабы. Дальше редактирование кода остаётся на странице, а тяжёлые прогоны упражнений и ★-челленджей исполняются здесь, на твоём рантайме.

Валидация — тот же harness, что и в песочнице браузера, поэтому результат (тесты, stdout, таймауты) идентичен байт-в-байт.

**Честные ограничения:** сессия Colab живёт ограниченное время (простой ≈ 90 минут, максимум ≈ 12 ч); quick-туннель живёт, пока работает ячейка, адрес меняется при перезапуске; бесплатный Colab — CPU. Это практика, никогда не evidence курса.

In [ ]:
# LSM companion runtime — единая ячейка настройки (run this cell first).
# 1) поднимает локальный HTTP-мост (только stdlib), исполняющий учебный harness лаб;
# 2) открывает публичный HTTPS-туннель (cloudflared quick tunnel, без аккаунта);
# 3) печатает URL и токен — вставь их в панель «Удалённый рантайм» на странице лабы.
import json, os, queue, re, secrets, subprocess, threading, time, urllib.request
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer

TOKEN = secrets.token_urlsafe(16)
PORT = 8765
MAX_BODY = 1024 * 1024
RUN_LOCK = threading.Lock()
RUNTIME_NAME = "lsm-colab-companion"

# Тот же harness, что исполняет песочница браузера (labs/py-worker.js):
# идентичные eq/approx, идентичный JSON-результат — валидация совпадает байт-в-байт.
HARNESS = r'''
import sys, io, json, traceback

class _Ring(io.TextIOBase):
    def __init__(self, cap=200):
        self.lines = []; self.cap = cap
    def write(self, s):
        s = str(s)
        for ln in s.splitlines(True):
            self.lines.append(ln.rstrip("\\n"))
            if len(self.lines) > self.cap:
                self.lines.pop(0)
        return len(s)
    def flush(self):
        pass

_ring = _Ring()
sys.stdout = _ring
sys.stderr = _ring
_M = {"expected": "expected", "got": "got"}

def _eq(got, expected):
    if got == expected:
        return
    raise AssertionError(_M["expected"] + " %r, " % (expected,) + _M["got"] + " %r" % (got,))

def _approx(got, expected, tol):
    try:
        ok = abs(got - expected) <= tol
    except TypeError:
        ok = False
    if ok:
        return
    raise AssertionError(_M["expected"] + " %r +/- %s, " % (expected, tol) + _M["got"] + " %r" % (got,))

def _dump():
    return "\\n".join(_ring.lines)

def _last(tb):
    lines = [l for l in tb.splitlines() if l.strip()]
    return lines[-1] if lines else tb

def _run_exercise(args_json):
    student_src, tests, msgs = json.loads(args_json)
    _M.update(msgs)
    _ring.lines.clear()
    ns = {"__name__": "__student__", "eq": _eq, "approx": _approx}
    try:
        exec(compile(student_src, "<student>", "exec"), ns)
    except SyntaxError:
        return json.dumps({"status": "syntax", "error": _last(traceback.format_exc(limit=3)),
                           "results": [], "stdout": ""})
    except BaseException:
        return json.dumps({"status": "error", "error": _last(traceback.format_exc(limit=4)),
                           "results": [], "stdout": _dump()})
    results = []
    for name, hidden, src in tests:
        try:
            exec(compile(src, "<test>", "exec"), ns)
            results.append({"name": name, "hidden": hidden, "passed": True, "kind": "", "message": ""})
        except AssertionError as e:
            results.append({"name": name, "hidden": hidden, "passed": False, "kind": "assert", "message": str(e)})
        except BaseException:
            results.append({"name": name, "hidden": hidden, "passed": False, "kind": "exception",
                            "message": _last(traceback.format_exc(limit=3))})
    status = "passed" if all(r["passed"] for r in results) else "failed"
    return json.dumps({"status": status, "error": "", "results": results, "stdout": _dump()})
'''

def _run_harness(payload):
    # Свежий форк на каждый прогон: зациклившийся студенческий код не клинит сервер.
    src = payload.get("code", "")
    tests = payload.get("tests", [])
    msgs = payload.get("msgs", {})
    args = json.dumps([src, tests, msgs])
    timeout_s = int(payload.get("timeout_s", 60))

    def work(q):
        try:
            ns = {}
            exec(compile(HARNESS, "<harness>", "exec"), ns)
            q.put(("ok", ns["_run_exercise"](args)))
        except BaseException:
            import traceback
            q.put(("fail", traceback.format_exc(limit=4)))

    try:
        import multiprocessing as mp
        ctx = mp.get_context("fork")
    except Exception:
        ctx = None
    if ctx is not None:
        q = ctx.Queue()
        p = ctx.Process(target=work, args=(q,), daemon=True)
        p.start()
        p.join(timeout_s)
        if p.is_alive():
            p.terminate(); p.join(2)
            return json.dumps({"status": "timeout", "error": "companion timeout", "results": [], "stdout": ""})
        if q.empty():
            return json.dumps({"status": "runner-error", "error": "companion worker died", "results": [], "stdout": ""})
        kind, data = q.get()
        if kind == "fail":
            return json.dumps({"status": "runner-error", "error": data, "results": [], "stdout": ""})
        return data
    # Нет fork (не-Linux): мягкий таймаут — брошенный поток может жить, но сервер отвечает.
    q = queue.Queue()
    t = threading.Thread(target=work, args=(q,), daemon=True)
    t.start()
    t.join(timeout_s)
    if t.is_alive():
        return json.dumps({"status": "timeout", "error": "companion timeout (no fork)", "results": [], "stdout": ""})
    if q.empty():
        return json.dumps({"status": "runner-error", "error": "companion worker died", "results": [], "stdout": ""})
    kind, data = q.get()
    if kind == "fail":
        return json.dumps({"status": "runner-error", "error": data, "results": [], "stdout": ""})
    return data

class Handler(BaseHTTPRequestHandler):
    protocol_version = "HTTP/1.1"

    def _cors(self):
        origin = self.headers.get("Origin") or "*"
        self.send_header("Access-Control-Allow-Origin", origin)
        self.send_header("Vary", "Origin")
        self.send_header("Access-Control-Allow-Methods", "GET, POST, OPTIONS")
        self.send_header("Access-Control-Allow-Headers", "Content-Type, X-Lab-Token")
        self.send_header("Access-Control-Allow-Private-Network", "true")
        self.send_header("Access-Control-Max-Age", "600")

    def _send(self, code, body, ctype="application/json"):
        self.send_response(code)
        self._cors()
        self.send_header("Content-Type", ctype)
        self.send_header("Content-Length", str(len(body)))
        self.end_headers()
        self.wfile.write(body)

    def _authed(self):
        return secrets.compare_digest(self.headers.get("X-Lab-Token", ""), TOKEN)

    def do_OPTIONS(self):
        self.send_response(204)
        self._cors()
        self.end_headers()

    def do_GET(self):
        if self.path != "/health":
            self._send(404, b'{"error":"not found"}'); return
        if not self._authed():
            self._send(401, b'{"error":"bad token"}'); return
        body = json.dumps({"status": "ok", "runtime": RUNTIME_NAME, "version": 1,
                           "caps": ["python3", "numpy"]}).encode()
        self._send(200, body)

    def do_POST(self):
        if self.path != "/run":
            self._send(404, b'{"error":"not found"}'); return
        if not self._authed():
            self._send(401, b'{"error":"bad token"}'); return
        n = min(int(self.headers.get("Content-Length", "0") or 0), MAX_BODY)
        try:
            payload = json.loads(self.rfile.read(n) or b"{}")
        except Exception:
            self._send(400, b'{"error":"bad json"}'); return
        if not RUN_LOCK.acquire(timeout=3):
            self._send(503, b'{"error":"busy - another run is in flight"}'); return
        try:
            result = _run_harness(payload)
        finally:
            RUN_LOCK.release()
        self._send(200, json.dumps({"ok": True, "result": result}).encode())

    def log_message(self, *a):
        pass

def start_tunnel(port):
    # cloudflared quick tunnel: без аккаунта; ngrok-альтернатива — см. ячейку ниже.
    url = None
    proc = None
    try:
        if not os.path.exists("./cloudflared"):
            print("downloading cloudflared (~40 MB)…")
            urllib.request.urlretrieve(
                "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
                "./cloudflared")
            os.chmod("./cloudflared", 0o755)
        proc = subprocess.Popen(
            ["./cloudflared", "tunnel", "--no-autoupdate", "--url", "http://127.0.0.1:%d" % port],
            stdout=subprocess.DEVNULL, stderr=subprocess.PIPE, text=True)
        t0 = time.time()
        while time.time() - t0 < 45 and url is None:
            line = proc.stderr.readline()
            if not line:
                time.sleep(0.2); continue
            mm = re.search(r"https://[-a-z0-9]+\.trycloudflare\.com", line)
            if mm:
                url = mm.group(0)
        if url is None:
            proc.terminate()
    except Exception as e:
        print("cloudflared failed:", repr(e))
    return proc, url

server = ThreadingHTTPServer(("127.0.0.1", PORT), Handler)
threading.Thread(target=server.serve_forever, daemon=True).start()
print("bridge up on 127.0.0.1:%d" % PORT)

proc, url = start_tunnel(PORT)
print("=" * 62)
if url:
    print("RUNTIME URL:", url)
else:
    print("Туннель не поднялся — запусти ячейку ещё раз (или см. ячейку ниже).")
print("TOKEN:      ", TOKEN)
print("Вставь URL и токен в панель «Удалённый рантайм» на странице лабы.")
print("=" * 62)


## Если что-то пошло не так

- **Туннель не поднялся** — запусти ячейку ещё раз: адрес quick-туннеля каждый раз новый.
- **Прогон завис** — Runtime → Interrupt execution, затем запусти ячейку заново.
- **Есть ngrok-токен** — можно заменить cloudflared:
  ```python
  !pip -q install pyngrok
  from pyngrok import ngrok
  ngrok.set_authtoken("…")
  print(ngrok.connect(8765, "http"))
  ```
  — полученный URL вставляется в панель лабы так же, вместе с токеном из ячейки выше.
- Мост слушает только 127.0.0.1 внутри Colab; наружу — только туннель, и доступ закрыт токеном. Не вставляй в лабу чужие адреса: рантайм исполняет тот код, который ты запускаешь.